# Simple: Email Generation with PAMOLA.CORE

**Goal**: Learn synthetic email generation basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Generate synthetic emails using execute()
- Compare before/after results
- Understand privacy and consistency benefits

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── fake_data/email/
        └── 01_fake_email_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.fake_data.operations.email_op import FakeEmailOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with records containing emails, names, and contact info

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with realistic email patterns
    sample_data = pd.DataFrame({
        'resume_id': range(1, 21),
        'email': [
            'john.smith@example.com', 'jane.doe@company.org', 'bob.wilson@gmail.com',
            'alice.jones@university.edu', 'mike.brown@startup.io', 'sarah.davis@enterprise.com',
            'tom.miller@yahoo.com', 'lisa.garcia@hotmail.com', 'david.rodriguez@outlook.com',
            'emma.martinez@protonmail.com', 'james.anderson@company.org', 'olivia.taylor@gmail.com',
            'william.thomas@example.com', 'sophia.jackson@university.edu', 'alexander.white@startup.io',
            'isabella.harris@enterprise.com', 'michael.martin@yahoo.com', 'charlotte.thompson@hotmail.com',
            'daniel.garcia@outlook.com', 'amelia.martinez@protonmail.com'
        ],
        'first_name': [
            'John', 'Jane', 'Bob', 'Alice', 'Mike', 'Sarah',
            'Tom', 'Lisa', 'David', 'Emma', 'James', 'Olivia',
            'William', 'Sophia', 'Alexander', 'Isabella', 'Michael', 'Charlotte',
            'Daniel', 'Amelia'
        ],
        'last_name': [
            'Smith', 'Doe', 'Wilson', 'Jones', 'Brown', 'Davis',
            'Miller', 'Garcia', 'Rodriguez', 'Martinez', 'Anderson', 'Taylor',
            'Thomas', 'Jackson', 'White', 'Harris', 'Martin', 'Thompson',
            'Garcia', 'Martinez'
        ],
        'department': [
            'Engineering', 'Sales', 'Marketing', 'Research', 'Engineering', 'Sales',
            'Marketing', 'HR', 'Engineering', 'Research', 'Sales', 'Marketing',
            'Engineering', 'Research', 'Engineering', 'Sales', 'Marketing', 'HR',
            'Engineering', 'Research'
        ]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure field name** for processing

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "email"  # Customize this!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "email"  # Field to process - CUSTOMIZE THIS!
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to process: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Sample values: {list(df[field_name].unique()[:5])}")

# Show domain distribution
domains = df[field_name].str.split('@').str[-1]
print(f"\n  Domain distribution:")
print(domains.value_counts().head())

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="fake_data",
    description="Synthetic email generation for privacy protection",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create FakeEmailOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `field_name='field_name'`: Field to process
- `mode='REPLACE'`: Modify original field (or 'ENRICH' for new field)
- `format='name_surname'`: Email format (e.g., name_surname, surname_name, nickname, existing_domain).
- `first_name_field='first_name'`: Use existing first names
- `last_name_field='last_name'`: Use existing last names
- `preserve_domain_ratio=0.5`: Keep 50% of original domains
- `consistency_mechanism='mapping'`: Deterministic generation
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = FakeEmailOperation(
    # Core parameters
    field_name=field_name,
    mode='REPLACE',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Email format parameters
    format='name_surname',            # Use name_surname format
    format_ratio=None,                # Use single format
    first_name_field='first_name',    # Reference to first name column
    last_name_field='last_name',      # Reference to last name column
    full_name_field=None,
    name_format=None,
    
    # Domain configuration
    domains=None,                     # Use default domain list
    preserve_domain_ratio=0.5,        # Keep 50% of original domains
    business_domain_ratio=0.2,        # 20% business domains
    
    # Email structure
    separator_options=['.', '_', ''], # Allowed separators
    number_suffix_probability=0.4,    # 40% chance of numeric suffix
    max_length=254,                   # RFC 5321 max length
    
    # Validation and handling
    validate_source=True,             # Validate input emails
    handle_invalid_email='generate_new',  # Generate new for invalid
    
    # Consistency mechanism
    consistency_mechanism='mapping',  # Use mapping for consistent replacement
    key='email-generation-key',       # Seed for consistency
    context_salt=None,                # Additional randomization
    
    # Mapping store (optional)
    save_mapping=True,               # Save mapping to file
    mapping_store_path=None,
    id_field='resume_id',             # For mapping consistency
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Metrics and output
    detailed_metrics=True,            # Collect detailed statistics
    max_retries=3,                    # Retry on generation failure
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field:              {operation.field_name}")
print(f"  Format:             {operation.format}")
print(f"  Consistency:        {operation.consistency_mechanism}")
print(f"  Domain preservation: {operation.preserve_domain_ratio:.0%}")
print(f"  Visualizations:     {operation.generate_visualization}")
print(f"  Save output:        {operation.save_output}")
print(f"  Force recalc:       {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing email generation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the processed output from task_dir
- Compare with original data
- Review metrics and artifacts

**Generated artifacts:**
- Processed CSV in output/
- Email mapping JSON in maps/
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Processed Records:")
    display_cols = [field_name]
    if 'resume_id' in result_df.columns:
        display_cols.insert(0, 'resume_id')
    if 'first_name' in result_df.columns:
        display_cols.append('first_name')
    if 'last_name' in result_df.columns:
        display_cols.append('last_name')
    print(result_df[display_cols].head(10))
    
    # Domain comparison
    print(f"\n📈 Domain Distribution (After Processing):")
    gen_domains = result_df[field_name].str.split('@').str[-1]
    domain_dist = gen_domains.value_counts()
    for domain, count in domain_dist.head(10).items():
        pct = (count / len(result_df)) * 100
        bar = '█' * int(pct / 5)
        print(f"  {domain:30s} {bar:20s} {count:2d} ({pct:5.1f}%)")
    
    # Email format analysis
    print(f"\n📧 Email Format Analysis:")
    has_dot = result_df[field_name].str.contains(r'\.', regex=True).sum()
    has_underscore = result_df[field_name].str.contains('_').sum()
    has_number = result_df[field_name].str.contains(r'\d', regex=True).sum()
    print(f"  With dot separator:       {has_dot} ({has_dot/len(result_df)*100:.1f}%)")
    print(f"  With underscore:          {has_underscore} ({has_underscore/len(result_df)*100:.1f}%)")
    print(f"  With numeric suffix:      {has_number} ({has_number/len(result_df)*100:.1f}%)")
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:   {len(df)}")
    print(f"  Processed records:  {len(result_df)}")
    print(f"  Original domains:   {df[field_name].str.split('@').str[-1].nunique()}")
    print(f"  Generated domains:  {gen_domains.nunique()}")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Synthetic emails protect privacy while maintaining realistic structure!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Synthetic names CSV
├── maps/             # Name mapping JSON
├── metrics/          # Generation metrics
├── reports/          # Detailed reports
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'maps', 'metrics', 'reports', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Email generation performance and effectiveness
2. **Output Data**: Synthetic email with sample rows
3. **Email Mapping**: Original → Synthetic email mappings
4. **Visualizations**: Charts showing email distribution and statistics

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'

if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))

    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Get latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)

        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            metadata = data.get("metadata", {})
            metrics = data.get("metrics", {})

            # -----------------------
            # METADATA
            # -----------------------
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            op = metadata.get("operation", {})
            print(f"   Operation: {op.get('class','N/A')}.{op.get('function','N/A')}")

            # -----------------------
            # BASIC METRICS
            # -----------------------
            print("\n📊 BASIC METRICS:")
            print(f"   Total Records: {metrics.get('total_records', 0):,}")
            print(f"   Non-null Records: {metrics.get('non_null_records', 0):,}")
            print(f"   Execution Time (operation-level): {metrics.get('execution_time', 0)}s")

            # -----------------------
            # PERFORMANCE
            # -----------------------
            if "performance" in metrics:
                perf = metrics["performance"]
                print("\n⚡ PERFORMANCE:")
                print(f"   Generation Time: {perf.get('generation_time', 0):.4f}s")
                print(f"   Records/Second: {perf.get('records_per_second', 0):,}")
                print(f"   Memory Usage: {perf.get('memory_usage_mb', 0):.2f} MB")

            # -----------------------
            # ORIGINAL DATA
            # -----------------------
            if "original_data" in metrics:
                od = metrics["original_data"]
                print("\n📩 ORIGINAL DATA:")
                print(f"   Total Records: {od.get('total_records', 0):,}")
                print(f"   Unique Values: {od.get('unique_values', 0):,}")

                vd = od.get("value_distribution", {})
                print(f"   Value Distribution (top 5):")
                for k, v in list(vd.items())[:5]:
                    print(f"      {k:<35} → {v:.4%}")

            # -----------------------
            # GENERATED DATA
            # -----------------------
            if "generated_data" in metrics:
                gd = metrics["generated_data"]
                print("\n✉️  GENERATED DATA:")
                print(f"   Total Records: {gd.get('total_records', 0):,}")
                print(f"   Unique Values: {gd.get('unique_values', 0):,}")

                # Top 5 distribution
                print(f"   Value Distribution (top 5):")
                for k, v in list(gd.get("value_distribution", {}).items())[:5]:
                    print(f"      {k:<40} → {v:.4%}")

                ls = gd.get("length_stats", {})
                if ls:
                    print("\n   Length Stats:")
                    print(f"      Min: {ls.get('min')}")
                    print(f"      Max: {ls.get('max')}")
                    print(f"      Mean: {ls.get('mean'):.2f}")
                    print(f"      Median: {ls.get('median')}")

            # -----------------------
            # TRANSFORMATION METRICS
            # -----------------------
            if "transformation_metrics" in metrics:
                tm = metrics["transformation_metrics"]
                print("\n🔁 TRANSFORMATION METRICS:")
                print(f"   Total Replacements: {tm.get('total_replacements', 0):,}")
                print(f"   Replacement Strategy: {tm.get('replacement_strategy', 'N/A')}")
                print(f"   Mapping Collisions: {tm.get('mapping_collisions', 0)}")
                print(f"   Reversibility Rate: {tm.get('reversibility_rate', 0):.2%}")
                print(f"   Distribution Similarity: {tm.get('distribution_similarity_score', 0):.3f}")
                print(f"   Uniqueness Preservation: {tm.get('uniqueness_preservation', 0):.2%}")

            # -----------------------
            # DICTIONARY METRICS
            # -----------------------
            if "dictionary_metrics" in metrics:
                dm = metrics["dictionary_metrics"]
                print("\n📚 DICTIONARY METRICS:")
                print(f"   Total Domains: {dm.get('total_domains', 0)}")
                pops = dm.get("popular_domains", [])
                print(f"   Popular Domains (top 5): {pops[:5]}")
                fd = dm.get("format_distribution", {})
                if fd:
                    print(f"   Format Distribution:")
                    for fmt, ratio in fd.items():
                        print(f"      {fmt:<20}: {ratio:.2%}")

            # -----------------------
            # GENERATOR (new)
            # -----------------------
            if "generator" in metrics:
                gen = metrics["generator"]
                print("\n🧬 GENERATOR:")
                print(f"   Type: {gen.get('type')}")
                print(f"   Consistency: {gen.get('consistency_mechanism')}")

            # -----------------------
            # EMAIL GENERATOR
            # -----------------------
            if "email_generator" in metrics:
                eg = metrics["email_generator"]
                print("\n📧 EMAIL GENERATOR:")
                print(f"   Format: {eg.get('format')}")
                print(f"   Domains Count: {eg.get('domains_count')}")
                print(f"   Validate Source: {eg.get('validate_source')}")
                print(f"   Invalid Handling: {eg.get('handle_invalid_email')}")

                nf = eg.get("name_fields_used", {})
                print("\n   Name Fields:")
                print(f"      First Name: {nf.get('first_name_field')}")
                print(f"      Last Name: {nf.get('last_name_field')}")
                print(f"      Full Name: {nf.get('full_name_field')}")
                print(f"      Name Format: {nf.get('name_format')}")

                # domain distribution block
                dd = eg.get("domain_distribution", {})
                print("\n   Domain Distribution:")
                print(f"      Total Emails: {dd.get('total_emails', 0):,}")
                print(f"      Unique Domains: {dd.get('unique_domains', 0)}")
                print(f"      Diversity Ratio: {dd.get('diversity_ratio', 0):.4f}")

                # categories
                dc = dd.get("domain_categories", {})
                if dc:
                    print("\n      Categories:")
                    for cat, ratio in dc.items():
                        print(f"         {cat.capitalize():<12}: {ratio:.2%}")

                # top domains
                td = dd.get("top_domains", {})
                if td:
                    print("\n      Top Domains:")
                    for domain, ratio in list(td.items())[:5]:
                        print(f"         {domain:<25}: {ratio:.2%}")

        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (NEWEST)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            # Email structure analysis
            if field_name in output_df.columns:
                print(f"\n\n📧 Email Structure Analysis:")
                print("-" * 80)
                emails = output_df[field_name].dropna()
                print(f"   Total Valid Emails: {len(emails)}")
                print(f"   Average Length: {emails.str.len().mean():.1f} characters")
                print(f"   With Dots: {emails.str.contains(r'\.').sum()} ({emails.str.contains(r'\.').sum()/len(emails)*100:.1f}%)")
                print(f"   With Underscores: {emails.str.contains('_').sum()} ({emails.str.contains('_').sum()/len(emails)*100:.1f}%)")
                print(f"   With Numbers: {emails.str.contains(r'\d', regex=True).sum()} ({emails.str.contains(r'\d', regex=True).sum()/len(emails)*100:.1f}%)")
                
                # Domain distribution
                print(f"\n\n📊 Domain Distribution:")
                print("-" * 80)
                domains = emails.str.split('@').str[-1]
                domain_counts = domains.value_counts()
                print(f"{'Domain':<40} {'Count':>6} {'Percentage':>12}")
                print("-" * 60)
                for domain, count in domain_counts.head(10).items():
                    pct = (count / len(domains)) * 100
                    print(f"{domain:<40} {count:>6} {pct:>11.2f}%")
        
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. EMAIL MAPPING (NEWEST)
print("\n\n3️⃣ EMAIL MAPPING:")
print("-" * 80)

maps_dir = task_dir / 'maps'
if not maps_dir.exists():
    print(f"⚠️  Maps directory not found: {maps_dir}")
else:
    mapping_files = list(maps_dir.glob('*_mapping.json'))
    
    if not mapping_files:
        print("⚠️  No mapping files found")
    else:
        # Sort by modification time (newest first)
        mapping_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_mapping_file = mapping_files[0]
        
        print(f"✓ Found {len(mapping_files)} mapping file(s)")
        print(f"📄 Loading LATEST: {latest_mapping_file.name}")
        mtime = datetime.fromtimestamp(latest_mapping_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_mapping_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            with open(latest_mapping_file, 'r', encoding='utf-8') as f:
                mapping_data = json.load(f)
            
            # Show mapping summary
            if 'mappings' in mapping_data:
                all_mappings = mapping_data['mappings']
                print(f"📋 Mapping Summary:")
                print(f"   Total Fields: {len(all_mappings)}")
                
                if field_name in all_mappings:
                    field_mappings = all_mappings[field_name]

                    print(f"   Total mappings for '{field_name}': {len(field_mappings)}")

                    print(f"\n📝 Sample Email Mappings (first 10):\n")
                    print(f"   {'Original Email':<40} → {'Synthetic Email':<40}")
                    print("   " + "-" * 90)

                    for row in field_mappings[:10]:
                        orig = row.get("original", "N/A")
                        synth = row.get("synthetic", "N/A")
                        print(f"   {orig:<40} → {synth:<40}")

                    if len(field_mappings) > 10:
                        print(f"\n   ... and {len(field_mappings) - 10} more mappings")
                else:
                    print(f"⚠️ Field '{field_name}' not found in mapping data")

            else:
                print("⚠️  No mappings found in file")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading mapping: {e}")

# 4. VISUALIZATIONS (NEWEST BATCH)
print("\n\n4️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            for i, viz_file in enumerate(latest_viz_batch, 1):
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)


## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with field config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review metrics and artifacts 

**Key takeaways:**
- operation.execute() handles end-to-end processing
- Execution behavior (visualizations, output saving, caching) configured in operation constructor
- Field name is configurable via create_config_kwargs()
- Synthetic emails protect privacy while maintaining realistic structure
- Name-based generation creates consistent, professional-looking addresses
- Domain preservation maintains organizational context
- PRGN consistency ensures reproducible generation
- All artifacts saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **02_fake_email_advanced.ipynb** includes:
- Multiple format strategies (nickname, first_last, initial_last)
- Format ratio distribution for mixed patterns
- Custom domain dictionaries and business domains
- Mapping store for persistent consistency
- Quality metrics and validation strategies
- Batch processing optimization
- Advanced error handling patterns

**Other Fake Data Operations:**
- 📗 **`01_fake_name_simple.ipynb`**: Generate synthetic person names 
- 📙 **`01_fake_phone_simple.ipynb`**: Generate synthetic phone numbers
- 📕 **`01_fake_organization_simple.ipynb`**: Generate synthetic organization names 

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Fake Data Generators Guide](../../docs/fake_data_guide.md)
- 🔒 [PRGN Consistency Guide](../../docs/prgn_mechanism.md)